# 第8回　仮説検定(1)：帰無仮説とt検定
## ―― 「差がある」とは、どういうことか

統計学Ⅰ（B）　／　北星学園大学

注目は、このコース屈指の誤解されやすい数 ――

> **p値は「効果の大きさ」でも「正しさの確率」でもない。**

### フック

> 新しい勉強法を試したクラスは、従来のクラスより**テスト平均が3点高かった**。
>
> これは「**新しい勉強法に効果があった**」のか、それとも「**たまたま**」なのか？

直感では決められない。3点は大きい？小さい？――この問いに答えるのが**仮説検定**だ。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan
df = df.dropna(subset=["通学時間min"])
print("準備OK")

---
## 1. 帰無仮説 ―― 背理法で考える

仮説検定は**背理法**に似ている。

1. まず「**差はない（偶然だ）**」と仮定する。これを **帰無仮説** という。
2. その仮定のもとで、**観測されたような差が起きる確率**を計算する。これが **p値**。
3. p値がとても小さい（＝偶然ではめったに起きない）なら、「差はない」という仮定のほうが間違っていた、と考える。これを「**有意差がある**」という。

> 「差がある」＝「**偶然では説明しにくいほどの差だった**」の言い換え。

---
## 2. t検定をやってみる（明確な差の例）

北辰大データで、**一人暮らしの人とそうでない人の通学時間**を比べる。2群の平均の差が偶然かを調べるのが **t検定**（`scipy.stats.ttest_ind`）。

In [ ]:
一人暮らし = df[df["一人暮らし"]=="はい"]["通学時間min"]
実家     = df[df["一人暮らし"]=="いいえ"]["通学時間min"]
print(f"一人暮らし: 平均 {一人暮らし.mean():.0f}分 (n={len(一人暮らし)})")
print(f"実家　　　: 平均 {実家.mean():.0f}分 (n={len(実家)})")

t, p = stats.ttest_ind(一人暮らし, 実家, equal_var=False)
print(f"\nt値 = {t:.1f}")
print(f"p値 = {p:.2e}")
print("→ p値は極めて小さい（0.05よりはるかに下）。")
print("  『差はない（偶然）』ではこの差はめったに起きない → 有意差あり。")

p値はほぼ0。「一人暮らしと実家で通学時間に差はない」と仮定すると、これほどの差（19分 vs 57分）は偶然ではまず起きない。だから**有意差あり**と判断する。

---
## 3. p値の意味を、正しく

ここで誤解を3つ、はっきり打ち消す。

- ❌ **p値は「帰無仮説が正しい確率」ではない**。p値は「帰無仮説が正しいと**仮定したとき**に、観測以上の差が出る確率」。
- ❌ **p値は「効果の大きさ」ではない**。p=0.001 でも差は小さいことがある。
- ❌ **「有意（p<0.05）＝重要」ではない**。有意は「偶然では説明しにくい」だけ。

これを、次のシミュレーションで体感する。

---
## 4. p値の脆さ ―― 同じ差でも n で変わる

「従来法58点」「新法60点」、**真の差はたった2点**で固定。ここから n 人ずつ取って t検定する、を2000回くりかえし、**有意（p<0.05）になった割合**を見る。

効果（2点）は同じなのに、n を増やすと…？

In [ ]:
rng = np.random.default_rng(2026)
mu1, mu2, sd = 58, 60, 11   # 真の差は2点で固定
for n in [20, 100, 500]:
    有意 = 0
    for _ in range(2000):
        a = rng.normal(mu1, sd, n)
        b = rng.normal(mu2, sd, n)
        if stats.ttest_ind(a, b)[1] < 0.05:
            有意 += 1
    print(f"n={n:3d}人/群: 有意(p<0.05)になった割合 = {有意/2000*100:3.0f}%")
print("\n真の差は常に2点。なのに n を増やすほど『有意』になりやすい。")
print("→ 『有意』は効果の大きさではない。n を増やせば小さな差でも有意にできる。")

n=20では約8%しか有意にならないのに、n=500では約80%。**真の効果（2点）は同じ**なのに、サンプルを増やすだけで「有意差あり」を作れてしまう。

だから「p<0.05だった！」だけでは、**効果が大きいことの証明にはならない**。効果の大きさは、別の指標で測る。

---
## 5. 効果量（Cohen's d）―― 「大きさ」を測る

p値とは別に、差の大きさ自体を標準化したのが **効果量 d**。「差が標準偏差いくつ分か」。目安：0.2小・0.5中・0.8大。

In [ ]:
def cohen_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.std(ddof=1)**2 + (nb-1)*b.std(ddof=1)**2) / (na+nb-2))
    return (a.mean() - b.mean()) / sp

print(f"通学時間の差（一人暮らし vs 実家）の効果量 d = {cohen_d(一人暮らし, 実家):.2f} → 大きい効果")
print(f"勉強法の差（58 vs 60）の効果量　　　　 d = {(60-58)/11:.2f} → 小さい効果")
print("\n勉強法の差は『有意にできる』が、効果量は小さい。有意 ≠ 効果が大きい。")

---
## 6. 2種類の誤り

検定は完璧ではない。間違え方が2つある。

| | 本当は差がない | 本当は差がある |
|---|---|---|
| 「差あり」と判定 | **第一種の誤り（偽陽性）** | 正解 |
| 「差なし」と判定 | 正解 | **第二種の誤り（見逃し）** |

- **有意水準 α＝0.05**：本当は差がないのに「差あり」と誤る確率を5%まで許す、という基準。
- だから「有意」でも、**20回に1回は偶然の偽陽性**かもしれない。何度も検定すれば偽陽性は増える（→第9回）。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 帰無仮説 | まず「差はない（偶然）」と仮定する |
| p値 | 帰無仮説のもとで観測以上の差が出る確率 |
| 有意差 | p<0.05 ＝「偶然では説明しにくい」 |
| ❌ よくある誤り | p値=帰無仮説が正しい確率／有意=効果大／n増やせば必ず有意 |
| 効果量 d | 差の大きさ。有意とは別に必ず見る |
| α・2種の誤り | 偽陽性5%を許す基準。検定の繰り返しに注意 |

> **「差がある」＝「偶然では説明しにくい」。それ以上でも以下でもない。**
> p値の小ささは効果の大きさを意味しない。**必ず効果量も見る。**

**課題（Moodle）**：t検定の出力を解釈し、「p<0.05だから効果が大きい」という主張の誤りを指摘する。